# PyHealth Tokenizer Tutorial

This notebook covers **`pyhealth.tokenizer`** — a lightweight utility for converting medical codes (or any string tokens) into integer indices for use with ML models.

You will learn:
- How to build a **`Vocabulary`** and use special tokens (`<pad>`, `<unk>`)
- How to wrap it in a **`Tokenizer`** for batch encoding/decoding
- How to encode flat lists of codes (**2D**) with padding and truncation
- How to encode visit-level patient histories (**3D**) for hierarchical models
- How to build a tokenizer from a real dataset's code vocabulary and feed it to a PyTorch model

> **Why a custom tokenizer?** Clinical codes (ICD, ATC, NDC, custom labels) are categorical and very sparse — a typical hospital uses only a few thousand of the ~70,000 valid ICD-10 codes. PyHealth's tokenizer is intentionally simple (no subword splitting, no BPE) because each medical code already *is* an atomic token. For free-text clinical notes, use a HuggingFace tokenizer instead.

---

In [ ]:
!pip install pyhealth

In [ ]:
from pyhealth.tokenizer import Vocabulary, Tokenizer

---
## Part 1: The `Vocabulary` Class

`Vocabulary` is the underlying token ↔ index mapping. You will rarely use it directly — `Tokenizer` wraps it — but it is useful to understand how special tokens are handled.

```python
Vocabulary(
    tokens: List[str],                       # the real tokens (ATC codes, ICD codes, drug names…)
    special_tokens: Optional[List[str]] = None,  # e.g. ['<pad>', '<unk>']
)
```

### Special token conventions

| Token | Purpose | When required |
|-------|---------|---------------|
| `<pad>` | Padding token for batch alignment | Required when `padding=True` in batch encoders |
| `<unk>` | Catch-all for unknown tokens | Required if your input may contain tokens outside the vocab; otherwise an unknown token raises `ValueError` |

**Order matters.** Special tokens are inserted *before* the real tokens, so `<pad>` will reliably get index `0` and `<unk>` will get index `1` when you pass them in that order. This convention is used throughout PyHealth's built-in models (e.g. embedding layers initialize the pad row to zero).

In [ ]:
# Build a vocabulary over the first 8 ATC level-3 codes
atc_codes = ['A01A', 'A02A', 'A02B', 'A02X', 'A03A', 'A03B', 'A03C', 'A03D']
vocab = Vocabulary(tokens=atc_codes, special_tokens=['<pad>', '<unk>'])

print(f'Vocab size: {len(vocab)}')
print(f'<pad> → {vocab("<pad>")}')
print(f'<unk> → {vocab("<unk>")}')
print(f'A01A  → {vocab("A01A")}')
print(f'A03D  → {vocab("A03D")}')

In [ ]:
# Containment check — useful when sanity-checking external inputs
print('"A03A" in vocab:', 'A03A' in vocab)
print('"Z99Z" in vocab:', 'Z99Z' in vocab)

# Calling the vocab with an unknown token falls back to <unk>
print('vocab("Z99Z") →', vocab('Z99Z'), ' (== index of <unk>)')

### What happens without `<unk>`?

If you build a vocab without `<unk>` and then query a token that is not in it, PyHealth raises an exception rather than silently mapping it to a default index. This is the safer default for supervised tasks where every input code should be known.

In [ ]:
strict_vocab = Vocabulary(tokens=atc_codes, special_tokens=['<pad>'])
try:
    strict_vocab('Z99Z')
except ValueError as e:
    print(f'ValueError: {e}')

---
## Part 2: The `Tokenizer` Class

`Tokenizer` wraps a `Vocabulary` and adds:
- single-list encode/decode
- 2D batch encode/decode (with padding + truncation)
- 3D batch encode/decode (visit-level, for hierarchical models)

```python
Tokenizer(
    tokens: List[str],
    special_tokens: Optional[List[str]] = None,
)
```

In [ ]:
# A more realistic ATC level-3 vocabulary (42 codes, covering ATC 'A' — alimentary tract)
token_space = [
    'A01A', 'A02A', 'A02B', 'A02X', 'A03A', 'A03B', 'A03C', 'A03D', 'A03E', 'A03F',
    'A04A', 'A05A', 'A05B', 'A05C', 'A06A', 'A07A', 'A07B', 'A07C', 'A07D', 'A07E',
    'A07F', 'A07X', 'A08A', 'A09A', 'A10A', 'A10B', 'A10X', 'A11A', 'A11B', 'A11C',
    'A11D', 'A11E', 'A11G', 'A11H', 'A11J', 'A12A', 'A12B', 'A12C', 'A13A', 'A14A',
    'A14B', 'A16A',
]

tokenizer = Tokenizer(tokens=token_space, special_tokens=['<pad>', '<unk>'])

print(f'Vocabulary size: {tokenizer.get_vocabulary_size()}')   # 2 specials + 42 codes = 44
print(f'Padding index : {tokenizer.get_padding_index()}')      # always 0 with this ordering

---
## Part 3: Converting Tokens ↔ Indices

For a single (1D) list of codes — e.g. one patient's diagnoses on one visit:

In [ ]:
tokens  = ['A03C', 'A03D', 'A03E', 'A03F', 'A04A', 'A05A', 'A05B', 'B035', 'C129']
#                                                                       ↑       ↑
#                                                       not in vocab — will map to <unk> (= 1)

indices = tokenizer.convert_tokens_to_indices(tokens)
print('tokens  → indices:', indices)

tokens_back = tokenizer.convert_indices_to_tokens(indices)
print('indices → tokens :', tokens_back)
print()
print('Note: "B035" and "C129" round-trip to "<unk>" — the original surface form is lost.')

---
## Part 4: 2D Batch Encoding (Padding + Truncation)

In practice you feed *batches* into a model. `batch_encode_2d` handles padding and truncation in one call.

```python
tokenizer.batch_encode_2d(
    batch:      List[List[str]],   # one inner list per sample
    padding:    bool = True,       # pad shorter lists to the batch maximum
    truncation: bool = True,       # truncate longer lists to max_length (keeps the LATEST tokens)
    max_length: int  = 512,
)
```

**Truncation note:** PyHealth keeps the *most recent* `max_length` tokens (`tokens[-max_length:]`). For longitudinal EHR sequences ordered oldest → newest, this preserves the most relevant recent history.

In [ ]:
batch = [
    ['A03C', 'A03D', 'A03E', 'A03F'],   # 4 codes
    ['A04A', 'B035', 'C129'],            # 3 codes (with two <unk>s)
]

# Default: pad to the batch maximum (4), no truncation needed
out1 = tokenizer.batch_encode_2d(batch)
print('default (padding=True, truncation=True):')
print(' ', out1)
print('  shape: 2 rows × 4 cols — second row padded with index 0 (<pad>)')

In [ ]:
# padding=False: rows keep their natural lengths (jagged — not directly usable as a tensor)
out2 = tokenizer.batch_encode_2d(batch, padding=False)
print('padding=False:')
print(' ', out2)

In [ ]:
# max_length=3 — forces truncation of the first row (which has 4 codes)
out3 = tokenizer.batch_encode_2d(batch, max_length=3)
print('max_length=3 (drops oldest token of row 0):')
print(' ', out3)
print()
print('Row 0 truncated from [A03C, A03D, A03E, A03F] → [A03D, A03E, A03F]')
print('Then row 1 has length 3 already — no padding needed, batch max is 3.')

### Decoding back to tokens

`batch_decode_2d` reverses the encode step. By default it strips `<pad>` tokens so you see only the real content. Pass `padding=True` to keep them visible (useful for debugging shape issues).

In [ ]:
indices = [
    [8, 9, 10, 11],
    [12, 1, 1, 0],   # ← the trailing 0 is <pad>
]

print('padding=False (default — drops <pad>):')
print(' ', tokenizer.batch_decode_2d(indices))
print()
print('padding=True (keep <pad> visible):')
print(' ', tokenizer.batch_decode_2d(indices, padding=True))

---
## Part 5: 3D Batch Encoding for Visit-Level Histories

EHR data is naturally **hierarchical**: a patient has multiple *visits*, each visit has multiple *codes*. Models like RETAIN, GAMENet, and SafeDrug expect input shaped as `[batch, visit, code]`.

`batch_encode_3d` pads and truncates along *both* axes:

```python
tokenizer.batch_encode_3d(
    batch:      List[List[List[str]]],
    padding:    Tuple[bool, bool] = (True,  True),   # (visits, codes)
    truncation: Tuple[bool, bool] = (True,  True),
    max_length: Tuple[int,  int]  = (10,    512),    # (max visits, max codes per visit)
)
```

| Argument | Axis 0 (visits) | Axis 1 (codes within a visit) |
|----------|-----------------|-------------------------------|
| `padding` | Pad batch to longest patient (visit count) | Pad to longest visit (code count) |
| `truncation` | Keep the most recent `max_length[0]` visits | Keep the most recent `max_length[1]` codes per visit |

The two axes are controlled independently — you can pad visits but not codes, etc.

In [ ]:
# Two patients with different visit counts and visit-level code counts
patients = [
    # Patient 0 — two visits
    [
        ['A03C', 'A03D', 'A03E', 'A03F'],   # visit 1 — 4 codes
        ['A08A', 'A09A'],                    # visit 2 — 2 codes
    ],
    # Patient 1 — one visit, with two unknown codes
    [
        ['A04A', 'B035', 'C129'],
    ],
]

out = tokenizer.batch_encode_3d(patients)
print('Default (pad both axes):')
for i, p in enumerate(out):
    print(f'  patient {i}: {p}')

In [ ]:
# pad visits but NOT codes within a visit — batch becomes jagged on the inner axis
out = tokenizer.batch_encode_3d(patients, padding=(True, False))
print('padding=(visits=True, codes=False):')
for i, p in enumerate(out):
    print(f'  patient {i}: {p}')

In [ ]:
# Truncate aggressively — max 2 visits, max 2 codes/visit
out = tokenizer.batch_encode_3d(patients, max_length=(2, 2))
print('max_length=(2 visits, 2 codes per visit):')
for i, p in enumerate(out):
    print(f'  patient {i}: {p}')
print()
print('Patient 0 visit 0 truncated from [A03C, A03D, A03E, A03F] → last 2 = [A03E, A03F]')
print('Patient 1 has only 1 visit, so the second visit slot is fully padded.')

### Decoding 3D batches

`batch_decode_3d` strips fully-padded visits by default. Pass `padding=True` to preserve the full rectangular shape — useful when you need the indices to align with attention masks or padding masks.

In [ ]:
indices_3d = [
    [
        [ 8,  9, 10, 11],
        [24, 25,  0,  0],   # ← two trailing pads
    ],
    [
        [12,  1,  1,  0],   # ← one trailing pad
        [ 0,  0,  0,  0],   # ← fully-padded visit (dropped by default)
    ],
]

print('padding=False (default — drops empty visits):')
for i, p in enumerate(tokenizer.batch_decode_3d(indices_3d)):
    print(f'  patient {i}: {p}')

print()
print('padding=True (full rectangular shape):')
for i, p in enumerate(tokenizer.batch_decode_3d(indices_3d, padding=True)):
    print(f'  patient {i}: {p}')

---
## Part 6: Practical Pattern — Build a Tokenizer from a Real Dataset

In a real training pipeline you don't hard-code the vocabulary — you build it from the codes that actually appear in your training set. The standard recipe:

1. Iterate over all training samples and collect the unique codes.
2. Sort them for reproducibility.
3. Build the tokenizer with `<pad>` and `<unk>` as special tokens.
4. Use the tokenizer's `get_vocabulary_size()` to size your embedding layer.

Below is a self-contained example you can run without downloading any data.

In [ ]:
# Simulate a small training set of patient visit histories
import random
random.seed(0)

# Pretend these are the ATC codes seen across the training corpus
corpus_codes = ['A01A', 'A02A', 'A02B', 'A03A', 'A03B', 'A04A', 'A05A',
                'A10A', 'A10B', 'A11A', 'A12A', 'B01A', 'B02B', 'C09A']

# 5 patients, each with 1–3 visits, each visit with 1–5 codes
train_samples = []
for _ in range(5):
    n_visits = random.randint(1, 3)
    visits = [random.sample(corpus_codes, k=random.randint(1, 5))
              for _ in range(n_visits)]
    train_samples.append(visits)

for i, p in enumerate(train_samples):
    print(f'patient {i}: {p}')

In [ ]:
# Step 1: collect the unique codes that actually appear in the training data
vocab_tokens = sorted({code for patient in train_samples
                            for visit in patient
                            for code in visit})
print(f'Unique codes in training data: {len(vocab_tokens)}')
print(vocab_tokens)

# Step 2: build the tokenizer
tokenizer = Tokenizer(tokens=vocab_tokens, special_tokens=['<pad>', '<unk>'])
print(f'\nFinal vocab size (incl. specials): {tokenizer.get_vocabulary_size()}')

In [ ]:
# Step 3: encode the full training set in one shot
encoded = tokenizer.batch_encode_3d(
    train_samples,
    padding=(True, True),
    truncation=(True, True),
    max_length=(5, 10),   # max 5 visits, max 10 codes per visit
)

print(f'Encoded batch — outer length: {len(encoded)} patients')
print(f'  visit count per patient: {[len(p) for p in encoded]}')
print(f'  codes per visit (patient 0): {[len(v) for v in encoded[0]]}')

In [ ]:
# Step 4 (sketch): use the tokenizer's vocab size as the embedding input dim
# This is what PyHealth's built-in models do internally:
#
#     import torch.nn as nn
#     embedding = nn.Embedding(
#         num_embeddings = tokenizer.get_vocabulary_size(),
#         embedding_dim  = 128,
#         padding_idx    = tokenizer.get_padding_index(),   # zero-row for <pad>
#     )
#
# Passing `padding_idx` ensures gradients don't flow into the pad row —
# critical for training stability when batches are heavily padded.

print(f'num_embeddings = {tokenizer.get_vocabulary_size()}')
print(f'padding_idx    = {tokenizer.get_padding_index()}')

---
## Part 7: Where the Tokenizer Fits in PyHealth

Most users never instantiate `Tokenizer` directly. It is used *internally* by:

| Component | Role of the tokenizer |
|-----------|----------------------|
| **`pyhealth.processors`** (`SequenceProcessor`, etc.) | Builds a tokenizer per feature key (e.g. one for diagnoses, one for procedures) when the dataset task is registered |
| **Built-in models** (`RNN`, `Transformer`, `RETAIN`, `GAMENet`, …) | Embeds tokenized inputs and uses `padding_idx` to mask the pad row |
| **`set_task()`** pipeline | Wires processors + tokenizers into the dataloader so models receive ready-to-embed integer tensors |

You should reach for `Tokenizer` directly when:
- You are writing a **custom model** that takes raw code lists instead of pre-processed tensors
- You are running **inference** on data that was not produced by PyHealth's pipeline (e.g. a CSV of codes)
- You need to **share a single vocabulary** across multiple datasets or tasks

For free-text inputs (clinical notes, discharge summaries), don't use this tokenizer — use a HuggingFace tokenizer inside one of PyHealth's text processors (see the `smart_processor_clinical_text_tutorial.ipynb` example for the canonical pattern).

---
## Summary

| Task | API |
|------|-----|
| Build a vocabulary | `Vocabulary(tokens, special_tokens=['<pad>', '<unk>'])` |
| Build a tokenizer  | `Tokenizer(tokens, special_tokens=['<pad>', '<unk>'])` |
| Get vocab size     | `tokenizer.get_vocabulary_size()` |
| Get pad index      | `tokenizer.get_padding_index()` |
| Encode one list    | `tokenizer.convert_tokens_to_indices(tokens)` |
| Decode one list    | `tokenizer.convert_indices_to_tokens(indices)` |
| Encode 2D batch    | `tokenizer.batch_encode_2d(batch, padding=True, truncation=True, max_length=512)` |
| Decode 2D batch    | `tokenizer.batch_decode_2d(batch, padding=False)` |
| Encode 3D batch    | `tokenizer.batch_encode_3d(batch, padding=(True, True), truncation=(True, True), max_length=(10, 512))` |
| Decode 3D batch    | `tokenizer.batch_decode_3d(batch, padding=False)` |

### Quick design checklist

- Always include `<pad>` if you will batch with `padding=True`.
- Include `<unk>` if you cannot guarantee a closed vocabulary at inference time.
- Order specials as `['<pad>', '<unk>']` so the pad index is `0` (matches PyHealth's model defaults).
- Build the vocab from the **training set only** — unseen codes at validation/test time should map to `<unk>`.
- For hierarchical EHR models, use `batch_encode_3d` with explicit `max_length` to cap GPU memory.